# Experiment 1.3.9 — Phase-aware Cross-user Contrastive Local Feature Learning

## Scientific question

Experiments 1.3.7–1.3.8 showed that the SNN local representation already contains task-relevant information complementary to Raw250 counts, while directly reconstructing raw count or early/late input statistics does not improve the SNN-only local representation.

This experiment therefore stops reconstructing raw input statistics and instead shapes the **geometry of the deployed SNN representation itself**. The hypothesis is that local SNN features should become more stable across users when local motion motifs with the same letter label and similar relative gesture phase are pulled together, while different letters in the same phase are separated.

The final deployment target remains SNN-only:

\[X_{spike} \rightarrow SNN \rightarrow z_1,z_2,\ldots,z_B, \qquad z_b\in\mathbb{R}^{64}.\]

No Raw branch and no projection head are kept at inference.

## Experimental conditions

- **A — `cls_only`**: whole-gesture CE only.
- **B — `con250`**: CE + phase-aware cross-user supervised contrastive loss directly on each 250 ms SNN local feature.
- **C — `con500`**: CE + the same contrastive loss on a 500 ms motif formed by concatenating two adjacent 250 ms local features. The deployed feature is still the original 250 ms 64-D feature.

Positive pairs are restricted to **same label + same relative-phase bucket + different user**. Negatives are restricted to **different label + same phase bucket**. Anchors without both a positive and a negative do not contribute to the contrastive loss. Only fully-valid windows participate; padded/partial windows are excluded.

For `con250`, the contrasted representation is `L2Normalize(log1p(z_b))`. For `con500`, it is `L2Normalize(log1p(concat[z_b,z_{b+1}]))`. There is deliberately **no projection head**, so the contrastive gradient shapes the representation that is actually deployed.

## Hyperparameter selection and leakage control

Temperature is fixed to `0.1`. The contrastive weight is selected from `0.01, 0.03, 0.10`. For each contrastive condition, all three training seeds `(11, 23, 101)` are run for every candidate lambda. The selected lambda maximizes **mean validation SNN250 fresh-probe balanced accuracy across the three seeds**. The test split is not accessed during this development stage.

Only after lambda selection do the final runs evaluate test performance.

In [1]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/01_setup.py

Repository root: /home/ted/project/writingring-viz
Experiment: experiment_1_3_9_phase_aware_cross_user_contrastive_local_features
Device: cuda
samples=853, users=20, classes=12
labels: ['A', 'B', 'C', 'D', 'E', 'G', 'H', 'I', 'J', 'K', 'L', 'X']
train/val/test: (632, 256, 30) (126, 256, 30) (95, 256, 30)
split: {'train_users': ('user_0', 'user_1', 'user_11', 'user_12', 'user_14', 'user_19', 'user_2', 'user_3', 'user_4', 'user_5', 'user_6', 'user_7', 'user_8', 'user_9'), 'val_users': ('user_15', 'user_16', 'user_20'), 'test_users': ('user_10', 'user_13', 'user_18')}
250 ms readout: 16 samples; bins= 16
contrastive temperature: 0.1
lambda grid: (0.01, 0.03, 0.1)


## Model and loss

The SNN architecture is unchanged from the previous local-feature experiments: `30 → 128 → 128 → 64`, with shifts `((2,3),(2,3),(2))`, `tau_mem=22.54 ms`, threshold `0.5`, continuous state through the full gesture, and no reset at 250 ms boundaries.

The total objective is

\[L = L_{cls} + \lambda_{con} L_{SupCon}.\]

The global classifier still reads the flattened sequence of 250 ms local features. The contrastive term is an auxiliary training regularizer only.

In [2]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/02_model_training.py

## Development sweep and final confirmation

Primary representation quality is measured with a frozen SNN and a fresh train-only-standardized Logistic Regression probe. `C` is selected on validation from `1e-3, 1e-2, 1e-1, 1, 10`.

Primary readout: **SNN250**. Secondary readout: **SNN125**, extracted from the same frozen L3 spike train to test whether improvements are representation-level rather than tied to one readout resolution.

In [3]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/03_run_experiment.py

/home/ted/project/writingring-viz/scripts/experiment_1_3_9_phase_aware_contrastive/02_model_training.py:76: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  ds = TensorDataset(*(torch.from_numpy(a) for a in arrays))


con250   seed= 11 lambda=0.010 epoch=  1 | train BA=0.0731 val BA=0.0833 | con=4.4873 anchors=0.995 | L3 FR=0.0000
con250   seed= 11 lambda=0.010 epoch= 10 | train BA=0.1656 val BA=0.2222 | con=5.6891 anchors=0.988 | L3 FR=0.1783
con250   seed= 11 lambda=0.010 epoch= 20 | train BA=0.1820 val BA=0.1708 | con=4.6923 anchors=0.990 | L3 FR=0.2904
con250   seed= 11 lambda=0.010 epoch= 30 | train BA=0.2423 val BA=0.1472 | con=4.5902 anchors=0.995 | L3 FR=0.3148
con250   seed= 11 lambda=0.010 epoch= 40 | train BA=0.2052 val BA=0.1574 | con=4.5580 anchors=0.991 | L3 FR=0.3199
con250   seed= 11 lambda=0.010 epoch= 50 | train BA=0.2376 val BA=0.1740 | con=4.5283 anchors=0.993 | L3 FR=0.3237
con250   seed= 11 lambda=0.010 epoch= 60 | train BA=0.2264 val BA=0.1957 | con=4.5189 anchors=0.992 | L3 FR=0.3293
con250   seed= 11 lambda=0.010 epoch= 70 | train BA=0.2848 val BA=0.1730 | con=4.4968 anchors=0.994 | L3 FR=0.3351
con250   seed= 11 lambda=0.010 epoch= 80 | train BA=0.2614 val BA=0.2278 | con=4

,condition,lambda_con,mean_probe_val_BA,sd_probe_val_BA,mean_global_val_BA,mean_valid_anchor_fraction
0,con250,0.01,0.240432,0.014168,0.237423,0.996499
1,con250,0.03,0.253428,0.019832,0.250882,0.996499
2,con250,0.10,0.273369,0.030968,0.283642,0.996499
3,con500,0.01,0.328649,0.016184,0.323302,0.982305
4,con500,0.03,0.289341,0.024054,0.291590,0.982305
5,con500,0.10,0.295789,0.033842,0.277282,0.982305


cls_only seed= 11 lambda=0.000 epoch=  1 | train BA=0.0748 val BA=0.0833 | con=0.0000 anchors=0.000 | L3 FR=0.0001
cls_only seed= 11 lambda=0.000 epoch= 10 | train BA=0.3168 val BA=0.1815 | con=0.0000 anchors=0.000 | L3 FR=0.0096
cls_only seed= 11 lambda=0.000 epoch= 20 | train BA=0.7003 val BA=0.3467 | con=0.0000 anchors=0.000 | L3 FR=0.0380
cls_only seed= 11 lambda=0.000 epoch= 30 | train BA=0.8254 val BA=0.4993 | con=0.0000 anchors=0.000 | L3 FR=0.0540
cls_only seed= 11 lambda=0.000 epoch= 40 | train BA=0.8853 val BA=0.5062 | con=0.0000 anchors=0.000 | L3 FR=0.0637
cls_only seed= 11 lambda=0.000 epoch= 50 | train BA=0.9223 val BA=0.5461 | con=0.0000 anchors=0.000 | L3 FR=0.0654
cls_only seed= 11 lambda=0.000 epoch= 60 | train BA=0.9567 val BA=0.6070 | con=0.0000 anchors=0.000 | L3 FR=0.0758
cls_only seed= 11 lambda=0.000 epoch= 70 | train BA=0.9741 val BA=0.6488 | con=0.0000 anchors=0.000 | L3 FR=0.0852
cls_only seed= 11 lambda=0.000 epoch= 80 | train BA=0.9794 val BA=0.6392 | con=0

KeyboardInterrupt: 

## Diagnostics

Two diagnostics are added without changing the target deployment architecture.

1. **Cross-user phase-aware retrieval/kNN**: query and candidate pool are constrained to different users and the same phase. This directly tests whether the intended cross-user geometry was created.
2. **Raw250 + SNN250 fusion**: diagnostic only. If SNN-only improves while fusion remains above Raw250, the representation became more robust without simply collapsing into a Raw-count copy.

Firing rates, dead-neuron fractions, and the fraction of contrastive-valid anchors are retained to diagnose pathological regimes.

In [ ]:
%run -i ../scripts/experiment_1_3_9_phase_aware_contrastive/04_diagnostics.py

## Decision rules

- If `con250` improves mean SNN250 BA and reduces seed variance, a 250 ms local feature is already a stable task-aware unit.
- If `con500` is stronger, individual 250 ms features likely act as primitives that become class-relevant only when composed into a short local motif.
- If both contrastive conditions underperform `cls_only`, letter identity is probably too strong a supervisory signal at this local timescale; the next step should move toward predictive/self-supervised local objectives rather than stronger class-conditioned local losses.
- Raw+SNN fusion is never treated as the final architecture; it is used only to diagnose retained complementarity.